In [12]:
import yfinance as yf
import pandas as pd
import time
import os

data_dir = '/Users/lawrence/Library/CloudStorage/GoogleDrive-lawrencemrmay@gmail.com/My Drive/_Uni/Stock_preds_LLM/data/'

# 1. Load the prices we just saved to get the successful tickers
df_prices = pd.read_parquet(
    os.path.join(data_dir, 'stocknet_10y_prices.parquet'),
    engine='pyarrow'
)

successful_tickers = df_prices['Symbol'].unique().tolist()

print(f"Extracting Yahoo Finance financials for {len(successful_tickers)} active tickers...")

all_fundamentals = []
failed_fundamentals = []

for i, ticker in enumerate(successful_tickers):
    print(f"[{i+1}/{len(successful_tickers)}] Pulling quarterly_cash_flow Statements for {ticker}...")
    
    try:
        stock = yf.Ticker(ticker)
        # Pull quarterly income statement and transpose it so Dates are rows
        df_statement = stock.quarterly_cash_flow.T
        
        if not df_statement.empty:
            df_statement = df_statement.reset_index()
            df_statement.rename(columns={'index': 'Filing_Date'}, inplace=True)
            df_statement['Symbol'] = ticker
            
            # Ensure the date is timezone-naive for merging later
            df_statement['Filing_Date'] = pd.to_datetime(df_statement['Filing_Date']).dt.tz_localize(None)
            
            all_fundamentals.append(df_statement)
        else:
            print(f"  -> Warning: No financial data found for {ticker}")
            failed_fundamentals.append(ticker)
            
    except Exception as e:
        print(f"  -> Error downloading {ticker}: {e}")
        failed_fundamentals.append(ticker)
        
    time.sleep(1) # Yahoo gets angry if you hit financials too fast

print("\n--- Fundamental Download Finished ---")

if all_fundamentals:
    master_fundamentals_df = pd.concat(all_fundamentals, ignore_index=True)
    master_fundamentals_df.to_csv(os.path.join(data_dir, 'stocknet_yfinance_quarterly_cash_flow_sheet.csv'), index=False)
    print(f"Success! Saved {len(master_fundamentals_df)} quarterly reports.")

Extracting Yahoo Finance financials for 82 active tickers...
[1/82] Pulling quarterly_cash_flow Statements for XOM...
[2/82] Pulling quarterly_cash_flow Statements for SHEL...
[3/82] Pulling quarterly_cash_flow Statements for CVX...
[4/82] Pulling quarterly_cash_flow Statements for TOT...
  -> Warning: No financial data found for TOT
[5/82] Pulling quarterly_cash_flow Statements for BP...
[6/82] Pulling quarterly_cash_flow Statements for BHP...
  -> Warning: No financial data found for BHP
[7/82] Pulling quarterly_cash_flow Statements for SNPMF...
  -> Warning: No financial data found for SNPMF
[8/82] Pulling quarterly_cash_flow Statements for SLB...
[9/82] Pulling quarterly_cash_flow Statements for AAPL...
[10/82] Pulling quarterly_cash_flow Statements for PG...
[11/82] Pulling quarterly_cash_flow Statements for BUD...
  -> Warning: No financial data found for BUD
[12/82] Pulling quarterly_cash_flow Statements for KO...
[13/82] Pulling quarterly_cash_flow Statements for PM...
[14/82] 